# AXI test 3: keep only the changes the request is about
Model: **Qwen2.5-7B-Instruct in 4-bit** (same as tests 1 and 2). Two parts, each with the rule stated in the prompt and without it:
* **Part 1, one step at a time** (about 4 min without the rule, 8 min with it): 140 requests of 7 kinds. Which policy ends with the right graph: *repair_m6* (what we had), *scope_m7* (new: only changes the request names are kept), and the older idea *scope_delta*.
* **Part 2, 30 requests in a row** (about 18 min with the rule, 7 min without it): does the held state stay right? *repair_m6* against *scope_m7*.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

If Colab disconnects: reconnect, run cells 1 to 4 again, then run the cell you were on again. It continues where it stopped.

The prediction and the reading rules were written before this run in `docs/SCOPE_TEST.md`.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-scope.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: load the model (downloads about 15 GB once; about 5 minutes)

In [ ]:
import sys, json, time, os, torch
sys.path.insert(0, "/content/axi/python")
from colab.colab_llm_experiment import build_hf_llm
from axi.experiments.long_horizon import run_long, summarize_long, print_long
from axi.experiments.scope_tasks import make_scope_tasks
from axi.experiments.scope_test import (run_scope_regime, summarize_scope, verdict_scope, verdict_scope_long, print_scope, print_exact, print_verdict)

MODEL = "Qwen/Qwen2.5-7B-Instruct"
llm, model = build_hf_llm(MODEL, "cuda", load_4bit=True)
os.makedirs("/content/out", exist_ok=True)
T0 = time.time()
def prog(i, n): print(f"  {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

### Cell 5: quick check, 14 requests and one episode of 4 steps (about 2 minutes). If it fails, stop and send me the red error text

In [ ]:
a = run_scope_regime(llm, make_scope_tasks(14, 7), True, "/content/smoke_single.jsonl", meta=dict(model=MODEL, smoke=True))
print_scope(a["summary"], "smoke, rule stated")
b = run_long(llm, "state", 1, 4, 99, True, "/content/smoke_long.jsonl", meta=dict(model=MODEL, smoke=True), policies=("repair_m6", "scope_m7"), mix="scope")
print_exact(b["summary"], "smoke, 4 steps")

### Cell 6: PART 1, rule stated in the prompt. 140 requests, one at a time (about 8 minutes)

In [ ]:
T0 = time.time()
tasks = make_scope_tasks(140, 7)
res_s_rule = run_scope_regime(llm, tasks, True, "/content/out/axi_scope_single_rule.jsonl", meta=dict(model=MODEL, seed=7), progress=prog)
print()
print_scope(res_s_rule["summary"], "PART 1, rule stated")
files.download(save(res_s_rule, "/content/out/axi_scope_single_rule.json"))

### Cell 7: PART 1, no rule in the prompt. 140 requests (about 4 minutes)

In [ ]:
T0 = time.time()
res_s_norule = run_scope_regime(llm, tasks, False, "/content/out/axi_scope_single_norule.jsonl", meta=dict(model=MODEL, seed=7), progress=prog)
print()
print_scope(res_s_norule["summary"], "PART 1, no rule")
print_verdict(verdict_scope({True: res_s_rule["summary"], False: res_s_norule["summary"]}), "PART 1, both regimes")
files.download(save(res_s_norule, "/content/out/axi_scope_single_norule.json"))

### Cell 8: PART 2, rule stated. 12 episodes x 30 requests, 2 policies (about 18 minutes; the counter shows episode i/12)

In [ ]:
T0 = time.time()
res_l_rule = run_long(llm, "state", 12, 30, 7, True, "/content/out/axi_scope_long_rule.jsonl", meta=dict(model=MODEL), policies=("repair_m6", "scope_m7"), progress=prog, mix="scope")
print()
print_long(res_l_rule["summary"], "PART 2, rule stated")
print_exact(res_l_rule["summary"], "PART 2, rule stated")
files.download(save(res_l_rule, "/content/out/axi_scope_long_rule.json"))

### Cell 9: PART 2, no rule. 8 episodes x 30 requests (about 7 minutes)

In [ ]:
T0 = time.time()
res_l_norule = run_long(llm, "state", 8, 30, 7, False, "/content/out/axi_scope_long_norule.jsonl", meta=dict(model=MODEL), policies=("repair_m6", "scope_m7"), progress=prog, mix="scope")
print()
print_long(res_l_norule["summary"], "PART 2, no rule")
print_exact(res_l_norule["summary"], "PART 2, no rule")
print_verdict(verdict_scope_long(res_l_rule["summary"], res_l_norule["summary"]), "PART 2, both regimes")
files.download(save(res_l_norule, "/content/out/axi_scope_long_norule.json"))

### Done
Files downloaded: `axi_scope_single_rule.json`, `axi_scope_single_norule.json`, `axi_scope_long_rule.json`, `axi_scope_long_norule.json`. Upload them to the chat, with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.